# Resume an Existing Atlas

This tutorial shows how to reopen an existing `.sasql` Atlas, inspect the
persisted analysis state, and continue a workflow in a later Python session
without importing the source data again.

Use this tutorial when you have already completed part of an analysis and want
to determine which results are available and which step should be run next.

By the end of this tutorial, you will be able to:

- safely reconnect to an existing Atlas;
- inspect imported data, metadata, and analysis results;
- review the current read index;
- identify the appropriate point from which to resume the workflow;
- continue analysis without unnecessarily recomputing existing results.

## Before You Begin

Resuming an Atlas restores information that has been written to the `.sasql`
database, including expression data, metadata, embeddings, cluster labels, and
other stored analysis results.

Objects that existed only in the previous Python session are not restored
automatically. These may include:

- Python variables other than data stored in the Atlas;
- active data iterators and minibatch generators;
- fitted scikit-learn or PyTorch models that were not saved separately;
- temporary arrays, data frames, and plotting objects.

Save external models and other Python objects separately when they are required
in a later session.


## 1. Verify the Atlas Path

Check that the expected `.sasql` file exists before constructing the `Atlas`
object:


In [ ]:
import os
from pathlib import Path
import scatlaspy as sap

os.chdir(Path("~/scAtlaspy-code-analysis").expanduser())

atlas_path = Path("./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(f"Atlas database not found: {atlas_path}")


Checking the path first helps prevent a misspelled path from being mistaken for
an existing analysis.


## 2. Open the Atlas

Create a new `Atlas` object using the same database path:


In [ ]:
atlas = sap.Atlas(
    atlas_path,
    db_memory_limit="8GB",
)

scAtlasPy connects to the existing database and makes its stored data and
analysis results available in the current Python session.

`db_memory_limit` configures the DuckDB connection created for this session. It
does not modify the expression data or analysis results already stored in the
database.


## 3. Inspect the Atlas Summary

Begin with a general summary:


In [ ]:
print(atlas)

List the available tables:


In [ ]:
atlas.table_names()

Preview the cell and gene metadata:


In [ ]:
atlas.head("obs", n=5)

Check that important metadata fields, such as sample, donor, batch, condition,
cluster, or cell-type annotations, are present.

You can inspect the schemas of the metadata tables with:


In [ ]:
atlas.table_info("obs")

In [ ]:
atlas.table_info("var")

## 4. Review the Persisted Workflow State

Use `workflow_state()` to summarize common artifacts that indicate how far the
analysis has progressed:


In [ ]:
atlas.workflow_state()


## 5. Inspect the Current Read Index

Many atlas-scale algorithms and streaming workflows operate through a read
index. The read index determines:

- which cells are included;
- which genes are included;
- whether highly variable genes are used;
- which expression field is read.

Inspect the persisted read-index configuration with:


In [ ]:
atlas.read_index_info()


Confirm that its cell selection, gene selection, and expression field match the
analysis you want to continue.

For example, a preprocessing workflow may use:


In [ ]:
sap.pp.scale(atlas, use_data="data_log1p", use_hvg=True)

atlas.build_read_index(
    cell_condition="filter_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_scale",
)


In this example:

- only cells passing `filter_cells` are included;
- only genes passing `filter_genes` are included;
- the selection is further restricted to highly variable genes;
- `scale()` creates a centered or standardized `data_scale` representation;
- downstream PCA and clustering methods read `data_scale`.

```{warning}
Do not rebuild the read index unless its current configuration is missing or
does not match the intended analysis.

Changing the read index does not automatically recompute existing PCA,
clustering, UMAP, or model results. Results calculated with an earlier cell
selection, gene selection, or expression field may no longer be consistent with
the new read index and should be recomputed when necessary.
```


## 6. Choose Where to Resume

Use the persisted state to identify the next analysis step.

| Current Atlas state | Typical next action |
|---|---|
| Expression data have been imported, but preprocessing is incomplete | Continue with quality control, filtering, normalization, and feature selection. |
| Preprocessing is complete, but no suitable read index exists | Construct the read index for the intended analysis. |
| The read index is ready, but PCA is missing | Run PCA. |
| PCA is available, but clustering is missing | Run the selected clustering method. |
| PCA is available, but UMAP is missing | Calculate UMAP coordinates. |
| Analysis results are complete | Inspect visualizations, query stored results, annotate cells, or export data. |

Run only the steps that are missing or that need to be recalculated.

For example, if preprocessing and read-index construction are complete but PCA
has not yet been calculated:


In [ ]:
# sap.tl.pca(
#     atlas,
#     n_components=30,
# )


If PCA is already available but distilled Louvain clustering is missing:


In [ ]:
# sap.tl.graph_clustering(
#     atlas,
#     mode="distilled_louvain",
#     add_obs_col="scatlas_cluster",
# )


If the required upstream results are already present, continue directly to the
corresponding downstream step rather than rerunning the entire workflow.

```{note}
Whether an existing result should be reused depends on more than its presence
in the database. Recompute downstream results when their upstream cell
selection, gene selection, expression representation, or parameters have
changed.
```


## 7. Close the Connection

Close the database connection when the current session is complete:


In [ ]:
atlas.close()


Closing the connection does not delete the `.sasql` file or its stored results.
The same Atlas can be reopened in another Python session using its database
path.

## Next Steps

- See {doc}`visualize-analysis-results` to inspect quality-control,
  dimensionality-reduction, clustering, and marker-analysis results.
- See {doc}`query-atlas-with-sql` to explore metadata and analysis results
  directly with SQL.
- Return to the {doc}`../basic/index` if quality control or preprocessing has
  not yet been completed.
